[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/04_gmm/GMM_and_Comparison.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/04_gmm/GMM_and_Comparison.ipynb)

# Notebook 04 — Gaussian Mixture Models + Full Comparison
## Soft Cluster Membership with Probability

**Dataset**: Iris flowers
**Question**: *What is the probability that each flower belongs to each species group?*
**Type**: Unsupervised — probabilistic clustering

---

## Section 1 — What Is a Gaussian Mixture Model?

### In plain English

K-Means gives every flower one hard assignment — "you belong to cluster 2, period."

A Gaussian Mixture Model (GMM) says: **"you probably belong to cluster 2 (75%) but you might
also belong to cluster 1 (25%)."**

Every cluster is modelled as a Gaussian (bell-curve) distribution in feature space.
Flowers in the core of a cluster get high probability. Flowers near the border get
shared probability across two clusters.

> This is called **soft assignment** — every point has a probability for every cluster.

### When GMM is useful

- When clusters overlap and a point legitimately could belong to two groups
- When cluster shapes are elliptical (not just spherical like K-Means assumes)
- When you need confidence scores, not just hard assignments

### GMM vs K-Means shape assumption

| | K-Means | GMM |
|---|---|---|
| Cluster shape | Spherical (equal radius) | **Elliptical (each cluster has its own shape)** |
| Assignment | Hard — one cluster | **Soft — probability per cluster** |
| Algorithm | Distance to centroid | **Expectation-Maximisation (EM)** |

## Section 2 — How Does It Learn?

### Expectation-Maximisation (EM) — two steps, alternated

**E-step (Expectation)**: for every flower, compute the probability it came from each Gaussian:
```
P(flower belongs to cluster k) ∝ weight_k × Gaussian_k(flower_features)
```

**M-step (Maximisation)**: update each Gaussian's mean, covariance, and weight
using the soft assignments from the E-step as "fractional membership counts"

**Repeat** until the parameters stop changing significantly.

> **Why is it called "mixture"?** The overall distribution of the data is modelled as a
> *mixture* of K Gaussians — a weighted sum of bell curves.
> The EM algorithm finds the K Gaussians that best explain the observed data.

## Section 3 — What Does the Data Need?

### Clustering is distance-based — scaling is essential

All clustering algorithms group points by how close they are to each other.
Distance is measured in feature space:
```
distance = sqrt((x1_a − x1_b)² + (x2_a − x2_b)² + ...)
```

If `sepal_length` ranges 4–8 cm and `petal_width` ranges 0.1–2.5 cm,
a 1 cm difference in sepal_length contributes equally to distance as a 1 cm difference in petal_width.
But the sepal feature has 4× the range — it dominates the distance calculation.

**Fix: StandardScaler** — all features end up with mean=0 and std=1.
Now every feature contributes equally to the distance.

### What clustering does NOT need

- No target labels (that is the whole point — we do not have them)
- No train/test split (there is no prediction to evaluate on held-out data)
- Only preparation: encode to numbers, fill missing values, scale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Load Iris — a classic dataset with 3 natural groups of flowers
df_raw = sns.load_dataset('iris')
FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
TRUE_LABELS = df_raw['species'].map({'setosa': 0, 'versicolor': 1, 'virginica': 2}).values

X_raw = df_raw[FEATURES].values

# Scale features — clustering algorithms are distance-based; scaling is required
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

print(f"Iris dataset: {X_raw.shape[0]} flowers, {X_raw.shape[1]} features")
print(f"True groups  : {df_raw['species'].value_counts().to_dict()}")
print()
print("We will PRETEND we do not know the species labels.")
print("The clustering algorithm must discover the 3 groups by itself.")
df_raw.head(6)

## Section 4 — Train GMM and Show Soft Assignments

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm.fit(X)

pred_labels = gmm.predict(X)
proba       = gmm.predict_proba(X)
sil         = silhouette_score(X, pred_labels)

print(f"Silhouette score: {sil:.4f}")
print()
print("Sample of soft membership probabilities (first 6 flowers):")
prob_df = pd.DataFrame(proba.round(3), columns=['P(cluster 0)','P(cluster 1)','P(cluster 2)'])
prob_df['Hard assignment'] = pred_labels
print(prob_df.head(6).to_string(index=False))
print()
print("Many flowers have probability > 0 in more than one cluster — soft assignment.")

In [ ]:
from matplotlib.patches import Ellipse

def draw_ellipse(pos, cov, ax, color, alpha=0.2, n_std=2):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=pos, width=w, height=h, angle=angle,
                  facecolor=color, alpha=alpha, edgecolor=color, linewidth=2)
    ax.add_patch(ell)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#5C6BC0', '#26A69A', '#FFA726']

# GMM with ellipses
for k in range(3):
    mask = pred_labels == k
    axes[0].scatter(X[mask,2], X[mask,3], color=colors[k], s=40, alpha=0.7,
                    label=f'Cluster {k}')
    cov = gmm.covariances_[k][2:, 2:][:, [0,1]][[0,1],:]
    draw_ellipse(gmm.means_[k][2:], cov, axes[0], colors[k])
axes[0].set_title(f'GMM Clusters (n=3, full covariance)\nSilhouette={sil:.3f}\nEllipses = cluster shape', fontsize=12)
axes[0].set_xlabel('petal_length (scaled)', fontsize=11)
axes[0].set_ylabel('petal_width (scaled)', fontsize=11)
axes[0].legend()

# True labels
for k, name in enumerate(['setosa','versicolor','virginica']):
    mask = TRUE_LABELS == k
    axes[1].scatter(X[mask,2], X[mask,3], color=colors[k], s=40, alpha=0.8, label=name)
axes[1].set_title('True Species', fontsize=12)
axes[1].set_xlabel('petal_length (scaled)', fontsize=11)
axes[1].set_ylabel('petal_width (scaled)', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 5 — All Four Clustering Methods Side by Side

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

methods = {
    'K-Means (K=3)'          : KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X),
    'DBSCAN (eps=0.5)'        : DBSCAN(eps=0.5, min_samples=5).fit_predict(X),
    'Hierarchical (Ward, K=3)': AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X),
    'GMM (n=3)'               : GaussianMixture(n_components=3, random_state=42).fit(X).predict(X),
}

rows = []
for name, labels in methods.items():
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (np.array(labels) == -1).sum()
    sil = silhouette_score(X, labels) if n_clusters > 1 and n_clusters < len(X)-1 else float('nan')
    rows.append({
        'Method'           : name,
        'Clusters found'   : n_clusters,
        'Noise points'     : n_noise,
        'Silhouette'       : round(sil, 4) if not np.isnan(sil) else 'N/A',
        'Needs K upfront'  : 'No' if name.startswith('DBSCAN') else 'Yes',
        'Handles noise'    : 'Yes' if name.startswith('DBSCAN') else 'No',
        'Soft assignment'  : 'Yes' if name.startswith('GMM') else 'No',
    })

results_df = pd.DataFrame(rows)
print("=== All Four Clustering Methods on Iris ===")
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
method_labels = list(methods.items())
titles = ['K-Means (K=3)', 'DBSCAN (eps=0.5)', 'Hierarchical\n(Ward, K=3)', 'GMM (n=3)']
colors = ['#5C6BC0', '#26A69A', '#FFA726', 'black']

for ax, (name, labels), title in zip(axes, method_labels, titles):
    unique = sorted(set(labels))
    col_map = {k: ('#333333' if k == -1 else ['#5C6BC0','#26A69A','#FFA726','#AB47BC'][i])
               for i, k in enumerate(u for u in unique if u != -1)}
    if -1 in unique:
        col_map[-1] = '#BBBBBB'
    for k in unique:
        mask = np.array(labels) == k
        lbl  = 'Noise' if k == -1 else f'Cluster {k}'
        marker = 'x' if k == -1 else 'o'
        ax.scatter(X[mask,2], X[mask,3], color=col_map[k], s=25, alpha=0.8,
                   marker=marker, label=lbl)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('petal_length', fontsize=9)
    ax.set_ylabel('petal_width', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Four Clustering Methods on Iris — petal_length vs petal_width',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print()
print('Key takeaways:')
print('  1. All clustering needs StandardScaler — algorithms measure distance')
print('  2. K-Means: simple, fast, spherical clusters, must specify K')
print('  3. DBSCAN: finds K automatically, handles noise, works on any shape')
print('  4. Hierarchical: no K needed upfront, shows full merge history (dendrogram)')
print('  5. GMM: soft probabilities, elliptical clusters, most flexible shape')